In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType, IntegerType
from pyspark.sql.functions import col, regexp_replace, nullif, lit, to_date
from pyspark.sql.types import DecimalType

def clean_decimal(c, p=38, s=8):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return nullif(cleaned, lit("")).alias(c)
def clean_int(c):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return when(
        length(cleaned) == 0,
        None
    ).otherwise(
        cleaned.cast(IntegerType())
    ).alias(c)
def clean_short(c):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return when(
        length(cleaned) == 0,
        None
    ).otherwise(
        cleaned.cast("short")
    ).alias(c)
def clean_date(c):
    return coalesce(
        expr(f"try_to_date({c}, 'M/d/yyyy')"),
        expr(f"try_to_date({c}, 'MM/dd/yyyy')"),
        expr(f"try_to_date({c}, 'yyyy-MM-dd')"),
        expr(f"try_to_date({c}, 'yyyyMMdd')")
    ).alias(c)
df = (
    spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .csv("/Volumes/mb_poc/data_raw/uc2/KHCN_HDV_D09_DGTAL_SVG.csv")
)
df_tgt = (
    df.select(
        clean_date("CDR_DT"),
        col("CST_LOB_NM").alias("CST_LOB_NM"),
        col("OU_CODE").alias("OU_CODE"),
        col("SUB_RGON").alias("SUB_RGON"),
        col("CST_GRP").alias("CST_GRP"),
        col("GRP_AGE").alias("GRP_AGE"),
        col("PD_TP").alias("PD_TP"),
        col("CHANNEL").alias("CHANNEL"),
        col("TERM_TP").alias("TERM_TP"),
        col("AR_RNG").alias("AR_RNG"),
        col("SPCL_INT_RATE_F").alias("SPCL_INT_RATE_F"),
        col("BRANCH_CODE").alias("BRANCH_CODE"),
        col("BRANCH_NAME").alias("BRANCH_NAME"),
        col("SUB_BRANCH_NAME").alias("SUB_BRANCH_NAME"),
        col("PRN_OU_TP").alias("PRN_OU_TP"),
        clean_decimal("NEW_OPN_TVR_TDY"),
        clean_int("NBR_OF_NEW_AC_TDY"),
        clean_decimal("CLS_BOOK_BAL"),
        clean_short("NBR_OF_NEW_AC_CST_TDY"),
        clean_int("NBR_OF_TVR_CST_TDY"),
        clean_decimal("NBR_OF_AC_W_BAL_TDY"),
        clean_decimal("NEW_OPN_TVR_YST"),
        clean_int("NBR_OF_NEW_AC_YST"),
        clean_decimal("CLS_BOOK_BAL_YST"),
        clean_int("NBR_OF_NEW_AC_CST_YST"),
        clean_int("NBR_OF_TVR_CST_YST"),
        clean_decimal("NEW_OPN_TVR_PRE_MO"),
        clean_int("NBR_OF_NEW_AC_PRE_MO"),
        clean_int("NBR_OF_NEW_AC_CST_PRE_MO"),
        clean_decimal("NEW_OPN_TVR_MTD"),
        clean_int("NBR_OF_NEW_AC_MTD"),
        clean_int("NBR_OF_NEW_AC_CST_MTD"),
        clean_decimal("NEW_OPN_TVR_LST_MO"),
        clean_int("NBR_OF_NEW_AC_LST_MO"),
        clean_decimal("CLS_BOOK_BAL_LST_MO"),
        clean_int("NBR_OF_NEW_AC_CST_LST_MO"),
        clean_int("NBR_OF_TVR_CST_LST_MO"),
        clean_int("NBR_OF_AC_W_BAL_LST_MO"),
        clean_decimal("CLS_BOOK_BAL_LST_YR"),
        clean_int("NBR_OF_TVR_CST_LST_YR"),
        clean_int("NBR_OF_AC_W_BAL_LST_YR")
    )
)
#df_tgt.printSchema()
#df_tgt.display()
#df_tgt.explain(True)

df_tgt.write.mode("append").insertInto("mb_poc.gold.rpt_khcn_hdv_d09_dgtal_svg")

